#Bước 1: Khai báo thư viện và Đọc dữ liệu JSON

In [ ]:
import json
import os
import matplotlib.pyplot as plt
import matplotlib.patches as patches
from PIL import Image
import requests
from io import BytesIO
from collections import defaultdict

# 1. Đọc file JSON (Thay đổi đường dẫn nếu file nằm ở thư mục khác)
json_path = 'swg_camera_traps.bounding_boxes.with_species.json'
with open(json_path, 'r', encoding='utf-8') as f:
    data = json.load(f)

# 2. Xây dựng các Dictionary để tra cứu nhanh
images_dict = {img['id']: img for img in data['images']}
categories_dict = {cat['id']: cat['name'] for cat in data.get('categories', [])}

# Gom nhóm annotations theo image_id
annotations_by_image = defaultdict(list)
for ann in data.get('annotations', []):
    annotations_by_image[ann['image_id']].append(ann)

# Tạo thư mục reports/ để lưu kết quả tự động
REPORTS_DIR = "../reports"
os.makedirs(REPORTS_DIR, exist_ok=True)

print(f"Tổng số ảnh: {len(images_dict)}")
print(f"Tổng số loài: {len(categories_dict)}")
print(f"Tổng số nhãn (bounding boxes): {len(data.get('annotations', []))}")

# Bước 2: Quét lỗi Bounding Box và Nhãn

In [ ]:
# Danh sách lưu trữ phân loại lỗi chi tiết
corrupted_images = []       # Ảnh bị đánh dấu corrupt trong metadata
missing_bbox_records = []   # Bản ghi không có trường 'bbox' (nhãn cấp ảnh)
bad_geometry_boxes = []     # Box thực sự bị lỗi hình học (kích thước <= 0 hoặc tràn viền)
multi_species_imgs = []     # Ảnh có chứa nhiều loài khác nhau

for img_id, img_info in images_dict.items():
    if img_info.get('corrupt', False):
        corrupted_images.append(img_id)
        continue

    anns = annotations_by_image.get(img_id, [])
    if not anns:
        continue

    img_w, img_h = img_info['width'], img_info['height']
    species_in_img = set()

    for ann in anns:
        if 'category_id' in ann:
            species_in_img.add(ann['category_id'])

        # 1. Kiểm tra bản ghi thiếu trường 'bbox'
        if 'bbox' not in ann or ann['bbox'] is None:
            missing_bbox_records.append({
                'image_id': img_id,
                'ann_id': ann.get('id')
            })
            continue

        bbox = ann['bbox']
        x, y, w, h = bbox

        # Chuyển đổi nếu tọa độ dạng chuẩn hóa (0.0 -> 1.0)
        if w <= 1.0 and h <= 1.0:
            x, y, w, h = x * img_w, y * img_h, w * img_w, h * img_h

        # 2. Kiểm tra lỗi hình học thực sự: kích thước <= 0 hoặc tràn biên ảnh
        # (Cho phép dung sai 1 pixel theo quy tắc làm sạch của TV1)
        if w <= 0 or h <= 0 or x < -1 or y < -1 or (x + w) > (img_w + 1) or (y + h) > (img_h + 1):
            bad_geometry_boxes.append({
                'image_id': img_id,
                'bbox': bbox,
                'reason': 'Kích thước <= 0 hoặc tọa độ tràn mép ảnh'
            })

    if len(species_in_img) > 1:
        multi_species_imgs.append(img_id)

print(f"1. Số ảnh đánh dấu hỏng (corrupt=True): {len(corrupted_images)}")
print(f"2. Số bản ghi không có trường 'bbox' (chỉ có nhãn cấp ảnh): {len(missing_bbox_records)}")
print(f"3. Số Bounding Box thực sự bị lỗi hình học (âm/tràn viền): {len(bad_geometry_boxes)}")
print(f"4. Số ảnh có từ 2 loài trở lên: {len(multi_species_imgs)}")

# Bước 3: Hàm vẽ Bounding Box (Kiểm tra mẫu hợp lệ)

In [ ]:
# Base URL của LILA cho dataset SWG Camera Traps
BASE_URL = "https://storage.googleapis.com/public-datasets-lila/swg-camera-traps/"
# Hoặc: "https://lilablobssc.blob.core.windows.net/swg-camera-traps/" (nếu link trên bị lỗi)

def plot_image_with_boxes(image_id, save_path=None):
    img_info = images_dict[image_id]
    file_name = img_info['file_name']
    url = BASE_URL + file_name

    # Tải ảnh từ URL
    response = requests.get(url)
    if response.status_code != 200:
        print(f"Không thể tải ảnh: {file_name}")
        return

    img = Image.open(BytesIO(response.content))
    fig, ax = plt.subplots(1, figsize=(10, 6))
    ax.imshow(img)

    anns = annotations_by_image.get(image_id, [])
    img_w, img_h = img_info['width'], img_info['height']

    for ann in anns:
        # BỎ QUA NẾU BẢN GHI KHÔNG CÓ BBOX (Tránh KeyError)
        if 'bbox' not in ann or ann['bbox'] is None:
            continue

        bbox = ann['bbox']
        cat_name = categories_dict.get(ann.get('category_id'), "Unknown")

        x, y, w, h = bbox
        # Chuyển tọa độ chuẩn hóa (nếu có) sang pixel
        if w <= 1.0 and h <= 1.0:
            x, y, w, h = x * img_w, y * img_h, w * img_w, h * img_h

        # Vẽ Rectangle (x_min, y_min, width, height)
        rect = patches.Rectangle((x, y), w, h, linewidth=2, edgecolor='red', facecolor='none')
        ax.add_patch(rect)
        ax.text(x, y - 10, str(cat_name), color='red', fontsize=12, backgroundcolor='white')

    plt.title(f"Image ID: {image_id}")
    plt.axis('off')

    # Tự động lưu ảnh vào reports/ nếu có đường dẫn
    if save_path:
        plt.savefig(save_path, bbox_inches='tight', dpi=150)
        print(f"  >> Da luu: {save_path}")

    plt.show()

# Danh sách các vị trí ảnh muốn kiểm tra
test_indices = [0, 3, 20]

# Tạo thư mục con để chứa ảnh kiểm tra bounding box
bbox_check_dir = os.path.join(REPORTS_DIR, "bbox_checks")
os.makedirs(bbox_check_dir, exist_ok=True)

for idx in test_indices:
    img_id = list(images_dict.keys())[idx]
    print(f"\n--- Đang hiển thị ảnh tại vị trí {idx} (ID: ...{img_id[-6:]}) ---")
    save_path = os.path.join(bbox_check_dir, f"check_bbox_pos{idx}_{img_id[-6:]}.png")
    plot_image_with_boxes(img_id, save_path=save_path)


In [ ]:
# Đếm chính xác số lượng Bounding Box thực tế (có tọa độ)
for idx in [0, 3, 20]:
    test_id = list(images_dict.keys())[idx]
    all_anns = annotations_by_image.get(test_id, [])

    # Lọc các annotation thực sự có chứa trường 'bbox'
    real_boxes = [ann for ann in all_anns if 'bbox' in ann and ann['bbox'] is not None]

    print(f"Vị trí {idx} (ID: ...{test_id[-6:]}): "
          f"{len(all_anns)} nhãn annotation, "
          f"{len(real_boxes)} bounding box thực tế")

### Đánh giá và Xử lý ngoại lệ dữ liệu Bounding Box

* **Hiện tượng dữ liệu:** Trong tập `annotations`, tồn tại nhiều bản ghi chỉ có nhãn phân loại cấp ảnh (`image_id`, `category_id`) mà **hoàn toàn không chứa khóa tọa độ `'bbox'`** (hoặc `bbox` mang giá trị `None`).
* **Xác thực qua ca kiểm thử (Ảnh vị trí số 3 - `c7d94921-8c29-11eb-bf65-000d3a74c7de`):**
  * Bản ghi `annotation` tồn tại (\(len = 1\)), nhưng số lượng bounding box thực tế bằng $0$.
  * Hàm `plot_image_with_boxes` đã xử lý an toàn bằng điều kiện lọc `if 'bbox' in ann and ann['bbox'] is not None:`, tải ảnh nền thành công và không bị dừng chương trình bởi lỗi `KeyError: 'bbox'`.
* **Khuyến nghị cho Pipeline Tuần 2 (W2-03):** Bộ nạp dữ liệu và kịch bản trích xuất vùng cắt (crop) bắt buộc phải kiểm tra sự tồn tại của khóa `'bbox'` trước khi cắt ảnh để tránh lỗi ngắt quãng và không gán nhãn toàn ảnh thành nhãn box sai lệch.

In [ ]:
from collections import Counter

# Đếm số lượng box hợp lệ cho từng loài
species_box_counts = Counter()
for anns in annotations_by_image.values():
    for ann in anns:
        if 'bbox' in ann and ann['bbox'] is not None:
            cat_id = ann.get('category_id')
            cat_name = categories_dict.get(cat_id, "Unknown")
            species_box_counts[cat_name] += 1

print("=== TOP CÁC LOÀI CÓ NHIỀU BOUNDING BOX NHẤT ===")
for name, count in species_box_counts.most_common(20):
    print(f"- {name}: {count} boxes")

# Bước 4: Lấy mẫu và hiển thị 10 ảnh cho mỗi loài mục tiêu

Đoạn mã này tự động lọc bỏ các ảnh không có bounding box, chọn đúng 10 ảnh hợp lệ cho từng loài trong 8 loài được chọn, và vẽ dạng lưới gọn gàng (2 hàng x 5 cột):

In [ ]:
!pip install ImageHash

In [ ]:
import requests
from PIL import Image
from io import BytesIO
import matplotlib.pyplot as plt
import matplotlib.patches as patches
import hashlib
import json
from collections import defaultdict

# 8 loài mục tiêu đã được TV1 chốt chính thức
TARGET_SPECIES = [
    'large_antlered_muntjac',
    'annamite_striped_rabbit',
    'sambar',
    'chinese_serow',
    'common_palm_civet',
    'masked_palm_civet',
    'silver_pheasant',
    'eurasian_wild_pig'
]

BASE_URL = "https://storage.googleapis.com/public-datasets-lila/swg-camera-traps/"

# Tạo ánh xạ ngược từ tên chuẩn hóa sang category_id
name_to_cat_id = {}
for cid, cname in categories_dict.items():
    norm_name = cname.lower().replace(" ", "_")
    name_to_cat_id[norm_name] = cid
    name_to_cat_id[cname.lower()] = cid

# Tạo các danh sách để ghi biên bản lỗi
error_log = {
    "missing_or_http_errors": [],
    "bad_boxes": [],
    "duplicates_found": []
}

downloaded_hashes = {} # Lưu trữ mã băm để kiểm tra trùng lặp

for species_name in TARGET_SPECIES:
    cat_id = name_to_cat_id.get(species_name)
    if not cat_id:
        print(f"Khong tim thay ID cho nhan {species_name}")
        continue

    # Tạo thư mục con cho từng loài trong reports/
    species_dir = os.path.join(REPORTS_DIR, species_name)
    os.makedirs(species_dir, exist_ok=True)

    # 1. GOM NHÓM ẢNH THEO ĐỊA ĐIỂM (LOCATION) ĐỂ TỐI ĐA HÓA SỰ ĐA DẠNG
    loc_to_imgs = defaultdict(list)
    for img_id, anns in annotations_by_image.items():
        has_valid_box = any(ann.get('category_id') == cat_id and 'bbox' in ann and ann['bbox'] is not None for ann in anns)
        if has_valid_box:
            loc = images_dict[img_id].get('location')
            loc_to_imgs[loc].append(img_id)

    # Lấy xoay vòng mỗi địa điểm 1 ảnh cho đến khi đủ 10 ảnh
    sampled_img_ids = []
    for loc, img_list in loc_to_imgs.items():
        sampled_img_ids.append(img_list[0])
        if len(sampled_img_ids) == 10:
            break

    print(f"\n[LOAI: {species_name.upper()}] - Da lay {len(sampled_img_ids)} mau tu cac dia diem doc lap.")

    fig, axes = plt.subplots(2, 5, figsize=(22, 8))
    axes = axes.flatten()

    for idx, img_id in enumerate(sampled_img_ids):
        ax = axes[idx]
        img_info = images_dict[img_id]
        url = BASE_URL + img_info['file_name']
        loc = img_info.get('location')
        seq = img_info.get('seq_id')

        try:
            resp = requests.get(url, timeout=15)
            if resp.status_code == 200:
                # 2. KIỂM TRA SHA-256 TRÙNG LẶP NỘI DUNG (Dùng Hashlib)
                image_bytes = resp.content
                sha256_hash = hashlib.sha256(image_bytes).hexdigest()

                if sha256_hash in downloaded_hashes:
                    error_log["duplicates_found"].append({
                        "img_id_1": downloaded_hashes[sha256_hash],
                        "img_id_2": img_id,
                        "sha256": sha256_hash
                    })
                downloaded_hashes[sha256_hash] = img_id

                img = Image.open(BytesIO(image_bytes))

                # Lưu ảnh đơn lẻ vào thư mục con của loài
                single_img_path = os.path.join(species_dir, f"{idx+1:02d}_{img_id[-8:]}.jpg")
                img.save(single_img_path)

                ax.imshow(img)
                img_w, img_h = img_info['width'], img_info['height']

                for ann in annotations_by_image.get(img_id, []):
                    if 'bbox' not in ann or ann['bbox'] is None:
                        continue

                    x, y, w, h = ann['bbox']
                    if w <= 1.0 and h <= 1.0:
                        x, y, w, h = x * img_w, y * img_h, w * img_w, h * img_h

                    # 3. PHÁT HIỆN BOX SAI KÍCH THƯỚC/TRÀN VIỀN (Ghi biên bản)
                    if w <= 0 or h <= 0 or x < -1 or y < -1 or x+w > img_w+1 or y+h > img_h+1:
                        error_log["bad_boxes"].append({"img_id": img_id, "bbox": [x,y,w,h]})
                    else:
                        rect = patches.Rectangle((x, y), w, h, linewidth=1.5, edgecolor='red', facecolor='none')
                        ax.add_patch(rect)

                ax.set_title(f"Loc: {loc}\nSeq: ...{seq[-4:]}", fontsize=9)
            else:
                error_log["missing_or_http_errors"].append({"img_id": img_id, "error": resp.status_code})
                ax.text(0.5, 0.5, 'Loi HTTP', ha='center', va='center')
        except Exception as e:
            error_log["missing_or_http_errors"].append({"img_id": img_id, "error": str(e)})
            ax.text(0.5, 0.5, 'Loi ket noi', ha='center', va='center')

        ax.axis('off')

    plt.suptitle(species_name.replace("_", " ").title(), fontsize=14, fontweight='bold')
    plt.tight_layout()

    # Lưu lưới 2x5 ảnh của loài vào thư mục loài
    grid_path = os.path.join(species_dir, f"grid_{species_name}.png")
    plt.savefig(grid_path, bbox_inches='tight', dpi=150)
    print(f"  >> Da luu luoi anh: {grid_path}")

    plt.show()

# 4. LƯU BIÊN BẢN LỖI vào reports/
error_log_path = os.path.join(REPORTS_DIR, "bien_ban_loi_w1.json")
with open(error_log_path, "w", encoding="utf-8") as f:
    json.dump(error_log, f, indent=4, ensure_ascii=False)

print("=== TONG KET BIEN BAN LOI (W1-03) ===")
print(f"- So anh loi tai / HTTP: {len(error_log['missing_or_http_errors'])}")
print(f"- So box sai kich thuoc/tran vien trong cac mau: {len(error_log['bad_boxes'])}")
print(f"- So anh trung lap SHA-256 phat hien duoc: {len(error_log['duplicates_found'])}")
print(f"Da luu chi tiet vao file '{error_log_path}'")


### Phát hiện giá trị từ việc khảo sát mẫu trực quan
Quá trình quét và xuất mẫu kiểm tra trực quan trên các địa điểm độc lập cho thấy các vấn đề đặc thù của dữ liệu bẫy ảnh cần lưu ý cho quá trình huấn luyện:

1. **Vấn đề tỷ lệ kích thước khung hình (Scale & Proximity Issue):**
   * **Hiện tượng:** Xuất hiện nhiều trường hợp động vật di chuyển quá sát ống kính camera, khiến bounding box mở rộng chiếm gần như toàn bộ diện tích bức ảnh.
   * **Ý nghĩa thực nghiệm:** Với các mẫu này, vùng cắt (crop image) có thông tin và bối cảnh gần tương đương với ảnh toàn cảnh (full image). Đây sẽ là nhóm mẫu đối chứng quan trọng khi so sánh hiệu năng giữa baseline toàn ảnh B1 và mô hình vùng cắt B2 ở Tuần 3.

2. **Dữ liệu đa cá thể trên một khung hình (Multi-instance Issue):**
   * **Hiện tượng:** Trong một số bức ảnh (điển hình ở các loài đi theo bầy như `eurasian_wild_pig`, `silver_pheasant`), xuất hiện đồng thời từ 2 cá thể trở lên và được gán các bounding box độc lập.
   * **Ý nghĩa thực nghiệm:** Phản ánh sự khác biệt về số lượng mẫu huấn luyện giữa hai hướng tiếp cận:
     * Đối với mô hình toàn ảnh (B1), toàn bộ khung hình chỉ đóng vai trò là **1 mẫu dữ liệu**.
     * Đối với mô hình vùng cắt (B2), bức ảnh này sẽ được phân tách thành **nhiều mẫu huấn luyện độc lập**, giúp gia tăng số lượng dữ liệu thực tế cho các lớp ít ảnh.
     * Cần lưu ý đánh chỉ số thứ tự cho từng crop ở Tuần 2 để tránh bị trùng tên file khi lưu trữ.

3. **Chất lượng ảnh hồng ngoại ban đêm (Night-time Overexposure):**
   * **Hiện tượng:** Các ảnh chụp ban đêm bằng cảm biến hồng ngoại thường xuyên gặp tình trạng chói sáng cục bộ (overexposed) do đèn flash đánh thẳng vào khoảng cách gần, làm mất chi tiết viền lông và hoa văn nhận dạng.
   * **Khuyến nghị kỹ thuật:** Cần bổ sung các phép tăng cường dữ liệu (Data Augmentation) về độ sáng, độ tương phản (ColorJitter/AutoContrast) trong pipeline huấn luyện ở Tuần 3 để tăng cường độ ổn định cho mô hình phân loại vùng cắt.

### Đối chiếu Tên khoa học và Tình trạng bảo tồn (Nguồn: Sách Đỏ IUCN)

Theo yêu cầu của đồ án về việc xác minh tính chuyên môn trước khi phân loại tình trạng quý hiếm, dưới đây là bảng tra cứu danh pháp khoa học và phân cấp bảo tồn IUCN cho 8 loài mục tiêu đã chọn.

| Tên nhãn (Dataset) | Tên tiếng Việt (Phổ thông) | Tên khoa học | Tình trạng bảo tồn (IUCN) | Ý nghĩa trong tập dữ liệu |
| :--- | :--- | :--- | :---: | :--- |
| `large_antlered_muntjac` | Mang lớn | *Muntiacus vuquangensis* | **CR** (Cực kỳ nguy cấp) | Loài đặc hữu Dãy Trường Sơn, đối tượng ưu tiên bảo tồn cao nhất. |
| `annamite_striped_rabbit`| Thỏ vằn Trường Sơn | *Nesolagus timminsi* | **EN** (Nguy cấp) | Loài đặc hữu Dãy Trường Sơn, số lượng mẫu phát hiện rất hiếm. |
| `sambar` | Nai | *Rusa unicolor* | **VU** (Sắp nguy cấp) | Thú móng guốc lớn, đối tượng thường xuyên bị săn trộm. |
| `chinese_serow` | Sơn dương lục địa | *Capricornis milneedwardsii* | **VU** (Sắp nguy cấp) | Thú móng guốc sống ở vách đá, sinh cảnh bị thu hẹp. |
| `common_palm_civet` | Cầy vòi hương | *Paradoxurus hermaphroditus*| **LC** (Ít quan tâm) | Lớp đối chứng đại diện cho nhóm thú ăn thịt nhỏ/cầy. |
| `masked_palm_civet` | Cầy vòi mốc | *Paguma larvata* | **LC** (Ít quan tâm) | Lớp dễ nhầm lẫn hình thái với cầy vòi hương, dùng để phân tích lỗi. |
| `silver_pheasant` | Gà lôi trắng | *Lophura nycthemera* | **LC** (Ít quan tâm) | Lớp đại diện cho nhóm chim đất/trĩ để đa dạng hóa tập phân loại. |
| `eurasian_wild_pig` | Lợn rừng | *Sus scrofa* | **LC** (Ít quan tâm) | Lớp phổ biến nhất, dùng làm baseline để kiểm tra mất cân bằng dữ liệu. |

**Kết luận:** Tập dữ liệu có sự phân hóa rõ rệt, bao gồm các nhóm loài hạt giống đặc hữu, cực kỳ nguy cấp (Mang lớn, Thỏ vằn) kết hợp với các nhóm kiểm soát phổ biến (Lợn rừng, Gà lôi). Việc sử dụng các nhãn này đã được đối chiếu, không tự ý gán mác "quý hiếm" cho các loài thuộc nhóm LC (Ít quan tâm).